<a href="https://colab.research.google.com/github/jugernaut/MACTI-manejodatos/blob/principal/07_RedesNeuronales/MetodoNewton_intercative.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
def f(x):
    """The function for which to find the root."""
    return x**2 - 2

def df(x):
    return 2*x

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches # Import patches for drawing rectangles
from ipywidgets import interact, FloatSlider, IntSlider
from IPython.display import display

def newton_with_steps(f, df, Tol, N, x0):
    """
    Newton's method that returns all intermediate approximations and tangent line info.
    f: The function.
    df: The derivative of the function.
    Tol: Tolerance for convergence.
    N: Maximum number of iterations.
    x0: Initial guess.
    Returns: (list of x approximations, list of tangent line info)
    """
    n = 1
    x_approximations = [x0] # Store initial guess
    tangent_lines_info = [] # Store info for tangent lines: (x_point, y_point, slope)

    current_x = x0

    while n <= N:
        fx = f(current_x)
        dfx = df(current_x)

        if dfx == 0:
            print(f"Warning: Derivative is zero at x = {current_x}. Newton's method cannot proceed further.")
            break

        # Calculate the next approximation
        next_x = current_x - (fx / float(dfx))

        # Store tangent line info for the current point
        tangent_lines_info.append({
            'x_point': current_x,
            'y_point': fx,
            'slope': dfx
        })

        x_approximations.append(next_x)

        # Check convergence criteria
        if abs(f(next_x)) <= Tol and abs(next_x - current_x) <= Tol:
            break

        current_x = next_x
        n += 1

    return x_approximations, tangent_lines_info

def plot_newton(iterations, initial_guess):
    """
    Garfica del metodo de Newton interactivo.
    iterations: Numero de iteraciones.
    initial_guess: Aproximacion inicial
    """
    tolerance = 1e-6

    # Get approximations and tangent info
    x_approxs, tangents = newton_with_steps(f, df, tolerance, iterations, initial_guess)

    plt.figure(figsize=(10, 7))

    # Determine x-axis range for plotting
    # Use a broader range if approximations are few or far apart
    x_min_plot = min(min(x_approxs) if x_approxs else initial_guess, -5.0) - 1
    x_max_plot = max(max(x_approxs) if x_approxs else initial_guess, 5.0) + 1

    # Ensure the root is within the visible range if found (for f(x) = x^2 - 2, roots are +/- sqrt(2))
    if len(x_approxs) > 1:
        root_approx = x_approxs[-1]
        x_min_plot = min(x_min_plot, root_approx - 2)
        x_max_plot = max(x_max_plot, root_approx + 2)
    else:
        x_min_plot = min(x_min_plot, initial_guess - 2)
        x_max_plot = max(x_max_plot, initial_guess + 2)

    x_vals = np.linspace(x_min_plot, x_max_plot, 400)
    y_vals = f(x_vals)

    # Set y-limits to fit the function and approximations well
    y_plot_min = y_vals.min()
    y_plot_max = y_vals.max()

    if x_approxs:
        y_approx_vals = [f(x) for x in x_approxs]
        y_plot_min = min(y_plot_min, min(y_approx_vals))
        y_plot_max = max(y_plot_max, max(y_approx_vals))

    # Add padding to y-limits
    y_plot_min -= 1
    y_plot_max += 1

    plt.plot(x_vals, y_vals, label='f(x) = $x^2 - 2$', color='blue', linewidth=2)
    plt.axhline(0, color='black', linewidth=1.2, linestyle='-', label='X-axis (y=0)') # x-axis
    plt.axvline(0, color='black', linewidth=1.2, linestyle='-', label='Y-axis (x=0)') # y-axis

    # Plot approximations and tangent lines
    for i in range(len(x_approxs)):
        x_val = x_approxs[i]
        y_val = f(x_val)

        # Plot approximation point on f(x)
        plt.scatter(x_val, y_val, color='red', s=70, zorder=5,
                    label=f'$x_{i}$ (Aprox. en f(x))' if i == 0 else "", marker='o', edgecolors='black')
        plt.text(x_val, y_val + 0.2, f'$x_{i}$', fontsize=10, ha='center', va='bottom')

        # Plot vertical line from approximation to x-axis
        plt.plot([x_val, x_val], [0, y_val], color='purple', linestyle=':', linewidth=1, alpha=0.7,
                 label='Residual (error))' if i == 0 else "_nolegend_")

        # Plot tangent line
        if i < len(tangents): # Tangents list is one shorter than x_approxs
            info = tangents[i]
            x_tangent_points = np.linspace(x_min_plot, x_max_plot, 100)
            y_tangent_points = info['slope'] * (x_tangent_points - info['x_point']) + info['y_point']

            plt.plot(x_tangent_points, y_tangent_points, color='green', linestyle='--', linewidth=1,
                     label=f'Tangente en $x_{i}$' if i == 0 else "_nolegend_", alpha=0.7)

            # Draw the point where the tangent intersects the x-axis (next approximation)
            if i < len(x_approxs) - 1: # If there's a next approximation
                next_x_val = x_approxs[i+1]
                plt.scatter(next_x_val, 0, color='orange', s=60, zorder=5, marker='x',
                            label=f'$x_{i+1}$ (siguiente aprox.)' if i==0 else "_nolegend_", edgecolors='black')
                plt.text(next_x_val, -0.4, f'$x_{i+1}$', fontsize=10, ha='center', va='top', color='orange')

    # Add evaluation of the last approximation of f(x)
    if x_approxs:
        final_x = x_approxs[-1]
        final_f_x = f(final_x)
        plt.text(x_max_plot - 0.5, y_plot_max - 0.5,
                 f'f($x_{len(x_approxs)-1}$): {final_f_x:.6f}',
                 fontsize=12, color='darkblue', ha='right', va='top',
                 bbox=dict(boxstyle="round,pad=0.3", fc="yellow", ec="b", lw=1, alpha=0.7))


    plt.title(f"Metodod de Newton Interactivo (Iteraciones: {iterations}, Iteracion Inicial: {initial_guess})")
    plt.xlabel('x')
    plt.ylabel('f(x)')
    plt.grid(True)

    # Custom legend to avoid duplicate labels from loop
    handles, labels = plt.gca().get_legend_handles_labels()
    unique_labels = dict(zip(labels, handles)) # Use dictionary to keep only unique labels
    plt.legend(unique_labels.values(), unique_labels.keys(), loc='best')

    plt.ylim(y_plot_min, y_plot_max)
    plt.xlim(x_min_plot, x_max_plot)
    plt.show()

# @title
# Create interactive sliders
interact(plot_newton,
         iterations=IntSlider(min=1, max=15, step=1, value=5, description='Iteraciones (N)', continuous_update=False),
         initial_guess=FloatSlider(min=-5.0, max=5.0, step=0.1, value=1.0, description='Iteracion Inicial $x_0$', continuous_update=False)
        );

interactive(children=(IntSlider(value=5, continuous_update=False, description='Iteraciones (N)', max=15, min=1…